# Task 1

## Importe

In [62]:
import base_net as bn
import pandapower as pp
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In der datei base_net.py wurde eine Funktion `build_base_net()` erstellt, mit der das Netz einfach erstellt werden kann für jeden test neu.

## Netzwerk
Externes Netz → Bus 1 (20 kV) → Transformator → Bus 2 (0,4 kV) → Kabel → Bus 3 (0,4 kV) → Last

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Mittelspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 2 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last | `load` an Bus 3 | P = 0,05 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

Alle Spannung werden in der Einheit `pu` (per unit) dargestellt. Diese ist bezogen auf den Bus.
### Vorteile
Die Vorteile daraus sind vor allem, das man direkt sieht, ob an einem Bus die entsprechende Spannung zu hoch, oder zu niedrig ist und kann so entsprechend regeln, ohne direkt zu wissen, wie hoch die spannung an dem entsprechenden Bus eigentlich sein sollte.

 Die Wirkleistung der Last wurde gegenüber dem Tutorial von 0,1 MW auf 0,05 MW reduziert. Mit 0,1 MW fließen etwa 148 A über das Kabel NAYY 4x50 SE, dessen thermisch zulässiger Dauerstrom 142 A beträgt (Auslastung 104,5 %). Ein Dauerbetrieb oberhalb von 100 % ist nicht zulässig. Mit 0,05 MW liegt die Auslastung bei 51 %, sodass in der Baseline ein realistischer, zulässiger Betriebspunkt vorliegt und Parametervariationen in beide Richtungen möglich sind.

## Datenstruktur

In [63]:
net, idx = bn.build_base_net()

In [64]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,20.0,b,None,True,None
1,Bus 2,0.4,b,None,True,None
2,Bus 3,0.4,b,None,True,None


In [65]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo,0.4 MVA 20/0.4 kV,0,1,0.4,20.0,0.4,6.0,1.425,1.35,...,0.0,0.0,Ratio,<NA>,False,1,1.0,True,Dyn5,False


In [66]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,1,2,0.1,0.642,0.083,210.0,0.0,0.142,1.0,1,cs,True,None


In [67]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load,2,0.05,0.005,0.0,0.0,0.0,0.0,NaN,1.0,True,wye


## Ausgangslage

In [68]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.005511
1,1.017429,-150.405971,0.000000,0.000000
2,0.997047,-150.439166,0.050000,0.005000


In [69]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051019,0.005131,-0.05,-0.005,0.001019,0.000131,0.072743,0.072744,0.072744,1.017429,-150.405971,0.997047,-150.439166,51.227854


In [70]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052513,0.005511,-0.051019,-0.005131,0.001494,0.00038,0.001494,0.072743,1.02,0.0,1.017429,-150.405971,12.941537


In [71]:
net.res_load

,p_mw,q_mvar
0,0.05,0.005


## Test 1 - trafo
tap_pos verringern und erhöhen

### Theorie


Der Stufensteller ändert die Übersetzung des Transformators in festen Stufen. Bei diesem Trafo gilt:

| Parameter | Wert | Bedeutung |
|---|---|---|
| `tap_side` | hv | Stufensteller auf der Oberspannungsseite |
| `tap_neutral` | 0 | Stellung mit Nennübersetzung 20/0,4 kV |
| `tap_min` / `tap_max` | −2 / +2 | zulässiger Stellbereich |
| `tap_step_percent` | 2,5 % | Spannungsänderung pro Stufe |
| `tap_changer_type` | Ratio | Längsregler: ändert nur den Betrag, nicht den Winkel |

Jede Stufe ändert die Windungszahl der OS-Wicklung um 2,5 %. Eine **positive** Stufe vergrößert die Übersetzung ü = U_OS / U_US. Bei fester OS-Spannung **sinkt** dadurch die US-Spannung. Eine **negative** Stufe hebt sie an. Der Stufensteller verschiebt das Spannungsniveau des gesamten Niederspannungsnetzes, ohne die Impedanzen nennenswert zu ändern.

### Erwartung

Wenn ich tap_pos erhöhe, erhöht sich die Windungszahl an der Oberspannungsseite, was dazu führt, dass an Bus 2 eine kleinere Spannung anliegt.

Dies wiederrum sogt dafür, dass ein höhrer Strom fliest, damit die Last die gleiche Leistung auifnimmt.

Dies sorgt dazu, dass an der Leitung eine höhere Spannung abfällt, weswegen dort die Auslastung steigt und an Bus 3 die Spannung noch weiter fällt, was widerrum die Ströme erhöht.

### tap_pos verringern

In [72]:
net, idx = bn.build_base_net()
net.trafo.at[idx["trafo"], "tap_pos"] = -1
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052529,-0.005485
1,1.043650,-150.385767,0.000000,0.000000
2,1.023801,-150.417282,0.050000,0.005000


In [73]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052529,0.005485,-0.050967,-0.005124,0.001562,0.000361,0.001495,0.070842,1.02,0.0,1.04365,-150.385767,12.944707


In [74]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.050967,0.005124,-0.05,-0.005,0.000967,0.000124,0.070842,0.070843,0.070843,1.04365,-150.385767,1.023801,-150.417282,49.889175


### tap_pos erhöhen

In [75]:
net, idx = bn.build_base_net()
net.trafo.at[idx["trafo"], "tap_pos"] = 1
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052504,-0.005538
1,0.992483,-150.426737,0.000000,0.000000
2,0.971566,-150.461658,0.050000,0.005000


In [76]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052504,0.005538,-0.051073,-0.005138,0.001431,0.0004,0.001494,0.074651,1.02,0.0,0.992483,-150.426737,12.940099


In [77]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051073,0.005138,-0.05,-0.005,0.001073,0.000138,0.074651,0.074651,0.074651,0.992483,-150.426737,0.971566,-150.461658,52.571358


## Test 2 - trafo
vk_percent verringern und erhöhen

### Theorie

`vk_percent` ist die **relative Kurzschlussspannung**: die Spannung, die bei kurzgeschlossener Unterspannungsseite nötig ist, damit Nennstrom fließt, bezogen auf die Nennspannung. Sie entspricht dem Betrag der Längsimpedanz des Transformators in pu, bezogen auf `sn_mva` = 0,4 MVA.

`vkr_percent` ist der ohmsche Anteil davon (Kupferverluste). Der induktive Anteil ist v_kx = √(v_k² − v_kr²) und entsteht durch das Streufeld der Wicklungen.

Ein größeres v_k bedeutet mehr lastabhängigen Spannungsfall, begrenzt aber den Kurzschlussstrom stärker. Für Ortsnetztransformatoren dieser Größe sind 4 % und 6 % übliche Werte. Daher wurde der Bereich 4–8 % gewählt. In diesem Versuch bleibt `vkr_percent` = 1,425 % konstant, es ändert sich also nur der induktive Anteil.

### Erwartung

### vk_percent verringern

In [78]:
net, idx = bn.build_base_net()
net.trafo.at[idx["trafo"], "vk_percent"] = 4
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.005374
1,1.017708,-150.256646,0.000000,0.000000
2,0.997332,-150.289822,0.050000,0.005000


In [79]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052513,0.005374,-0.051019,-0.005131,0.001494,0.000244,0.001494,0.072723,1.02,0.0,1.017708,-150.256646,12.938027


### vk_percent erhöhen

In [80]:
net, idx = bn.build_base_net()
net.trafo.at[idx["trafo"], "vk_percent"] = 8
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.005645
1,1.017149,-150.552025,0.000000,0.000000
2,0.996761,-150.585238,0.050000,0.005000


In [81]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.052513,0.005645,-0.05102,-0.005131,0.001494,0.000514,0.001495,0.072764,1.02,0.0,1.017149,-150.552025,12.945057


## Test 3 - line
r_ohm_per_km verringern und erhöhen - kabellänge: 100m

### Theorie

`r_ohm_per_km` ist der **Wirkwiderstand** der Leitung pro Kilometer im π-Ersatzschaltbild. Der Gesamtwiderstand ergibt sich aus

R = r · `length_km` / `parallel` = 0,642 Ω/km · 0,1 km / 1 = 0,0642 Ω.

R verursacht Stromwärmeverluste (P_V = 3 · I² · R) und zusammen mit der Wirkleistung den Spannungsfall. Er hängt vom Leitermaterial und vom Querschnitt ab: Ein größerer Querschnitt bedeutet einen kleineren Widerstand. Der Bereich 0,5–0,8 Ω/km entspricht ungefähr den benachbarten Querschnitten eines Aluminiumkabels.

`max_i_ka` ist nur die thermische Stromgrenze und hat keinen Einfluss auf die Impedanz.

### Erwartung

Wenn ich den Wiederstand an der Leitung erhöhe fällt dort eine höhrere Spannung ab, weswegen an Bus 3 eine geringere Spannung herscht.

Dies sorgt dazu, dass ein höhere Strom fliest, damit die Last die gleiche Leistung aufnimmt. Dies erhöht wieder rum die abfallende Spannung an der Leitung, was wiederrum die die ströme erhöht.

### r_ohm_per_km verringern

In [82]:
net, idx = bn.build_base_net()
net.line.at[idx["line"], "r_ohm_per_km"] = 0.5
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.05228,-0.005506
1,1.017438,-150.404099,0.00000,0.000000
2,1.001578,-150.462091,0.05000,0.005000


In [83]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.050787,0.005129,-0.05,-0.005,0.000787,0.000129,0.072414,0.072414,0.072414,1.017438,-150.404099,1.001578,-150.462091,50.99611


### r_ohm_per_km erhöhen

In [84]:
net, idx = bn.build_base_net()
net.line.at[idx["line"], "r_ohm_per_km"] = 0.8
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052778,-0.005516
1,1.017419,-150.408096,0.000000,0.000000
2,0.991955,-150.413433,0.050000,0.005000


In [85]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051283,0.005132,-0.05,-0.005,0.001283,0.000132,0.073117,0.073117,0.073117,1.017419,-150.408096,0.991955,-150.413433,51.490808


## Test 4 - line
x_ohm_per_km verringern und erhöhen - kabellänge: 100m

### Theorie

`x_ohm_per_km` ist der **induktive Blindwiderstand** pro Kilometer (X = ωL). Er hängt vor allem vom Abstand der Leiter ab. Bei Kabeln liegen die Leiter eng beieinander, deshalb ist X klein (typisch 0,07–0,09 Ω/km). Freileitungen liegen bei etwa 0,3–0,4 Ω/km.

X bestimmt den Blindleistungsbedarf der Leitung (I² · X) und vor allem die Winkeldifferenz zwischen den Leitungsenden. Die Querkapazität `c_nf_per_km` ist bei 100 m Niederspannungskabel vernachlässigbar.

### Erwartung

### x_ohm_per_km verringern

In [86]:
net, idx = bn.build_base_net()
net.line.at[idx["line"], "x_ohm_per_km"] = 0.04
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.005443
1,1.017439,-150.406099,0.000000,0.000000
2,0.997194,-150.363404,0.050000,0.005000


In [87]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051019,0.005062,-0.05,-0.005,0.001019,0.000062,0.072733,0.072733,0.072733,1.017439,-150.406099,0.997194,-150.363404,51.22028


### x_ohm_per_km erhöhen

In [88]:
net, idx = bn.build_base_net()
net.line.at[idx["line"], "x_ohm_per_km"] = 0.12
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.052513,-0.00557
1,1.017421,-150.405861,0.000000,0.00000
2,0.996918,-150.504375,0.050000,0.00500


In [89]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051019,0.005189,-0.05,-0.005,0.001019,0.000189,0.072753,0.072753,0.072753,1.017421,-150.405861,0.996918,-150.504375,51.23445


## Test 5 - load
p_mw verringern und erhöhen

### Theorie

`p_mw` ist die **Wirkleistung** der Last in MW. pandapower verwendet das **Verbraucherzählpfeilsystem**: Ein positiver Wert bedeutet Leistungsbezug, ein negativer Wert Einspeisung.

Da alle ZIP-Anteile (`const_z_p_percent`, `const_i_p_percent`) 0 sind, ist die Last eine **Konstantleistungslast**: Sie bezieht unabhängig von der Spannung immer genau P. Sinkt die Spannung, steigt der Strom (I = P / (√3·U)). Dadurch wachsen Spannungsfall und Verluste überproportional. Oberhalb der maximal übertragbaren Leistung gibt es keine Lösung der Leistungsflussgleichungen mehr, und der Lastfluss konvergiert nicht.

### Erwartung

Wenn die Leistung erhöht wird fliest ein höherer Strom, weswegen an der Leitung eine höhrere Spannung abfällt und auch eien höhere Leistung verbraucht wird.

Dies sorgt dafür, das an Bus3 die Spannung geringer ist, weswegen ein noch höherer Strom fliest.

Das ganze sollte sich ein pendeln, sofern die Leistung aufnahme der Last klein genug bleibt.

Außerdem sollte die prozentuale Auslastung steigen.

Wenn ich die aufgenomene Leistung der Last verringere sollte ein genau gegenteiliger Effekt zu sehen sein.

### p_mw verringern

In [90]:
net, idx = bn.build_base_net()
net.load.at[idx["load"], "p_mw"] = 0.02
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.021585,-0.005085
1,1.018548,-150.157826,0.000000,0.000000
2,1.010348,-150.103902,0.020000,0.005000


In [91]:
net.res_load

,p_mw,q_mvar
0,0.02,0.005


In [92]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.020167,0.005021,-0.02,-0.005,0.000167,0.000021,0.029451,0.029451,0.029451,1.018548,-150.157826,1.010348,-150.103902,20.740271


### p_mw erhöhen - 1

In [123]:
net, idx = bn.build_base_net()
net.load.at[idx["load"], "p_mw"] = 0.6
try:
    pp.runpp(net)
    net.res_bus
except Exception as e:
    print(f"Fehler-Typ und Text: {repr(e)}")

Fehler-Typ und Text: LoadflowNotConverged('Power Flow nr did not converge after 10 iterations!')


In [94]:
net.res_load

,p_mw,q_mvar
0,0.5,0.005


In [95]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.708003,0.031891,-0.5,-0.005,0.208003,0.026891,1.039217,1.039217,1.039217,0.984347,-155.837841,0.694488,-157.844001,731.842921


### p_mw erhöhen - 2

In [96]:
net, idx = bn.build_base_net()
net.load.at[idx["load"], "p_mw"] = 0.2
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.221905,-0.014315
1,1.010709,-151.763296,0.000000,0.000000
2,0.923489,-152.277032,0.200000,0.005000


In [97]:
net.res_load

,p_mw,q_mvar
0,0.2,0.005


In [98]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.218831,0.007434,-0.2,-0.005,0.018831,0.002434,0.312689,0.31269,0.31269,1.010709,-151.763296,0.923489,-152.277032,220.203877


## Test 6 - load
q_mvar verringern und erhöhen

### Theorie

`q_mvar` ist die **Blindleistung** der Last in Mvar. Im Verbraucherzählpfeilsystem gilt:

- Q > 0: Die Last bezieht Blindleistung und verhält sich **induktiv** (z. B. Motoren).
- Q < 0: Die Last gibt Blindleistung ab und verhält sich **kapazitiv** (z. B. Kompensationsanlage).

Blindleistung überträgt keine Energie, verursacht aber Strom und damit Verluste. Über den Term X·Q wirkt sie vor allem dort auf den Spannungsbetrag, wo die Netzimpedanz induktiv ist, hier also am Transformator.

### Erwartung

### q_mvar verringern

In [99]:
net, idx = bn.build_base_net()
net.load.at[idx["load"], "q_mvar"] = -0.1
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.056708,0.0976
1,1.032050,-150.633395,0.000000,0.0000
2,1.016561,-152.967009,0.050000,-0.1000


In [100]:
net.res_load

,p_mw,q_mvar
0,0.05,-0.1


### q_mvar erhöhen

In [101]:
net, idx = bn.build_base_net()
net.load.at[idx["load"], "q_mvar"] = 0.1
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.057108,-0.102598
1,1.003384,-150.255539,0.000000,0.000000
2,0.976798,-148.060977,0.050000,0.100000


In [102]:
net.res_load

,p_mw,q_mvar
0,0.05,0.1


## Aufgabe 6.1

Vorgelagerten Netzes 110kV

### Netzwerk

Externes Netz → Bus 2 (110 kV) →  Transformator → Bus 2 (20 kV) → Transformator → Bus 3 (0,4 kV) → Kabel → Bus 4 (0,4 kV) → Last

Zwischen dem Externen Netz und dem Bus des Mittelspannungsnetz wurde ein enuer Bus und ein neuer Bus `bus1 110kv` und ein Transformator `25 MVA 110/20 kV` hinzugefügt.

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Mittelspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 110 kV | Nennspannung 110 kV | Hochspannung-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator 1 | `trafo`, Typ „25 MVA 110/20 kV“ | S_r = 25 MVA, v_k = 12 %, v_kr = 0,41 %, Schaltgruppe Dyn5, Stufensteller ±9 × 1,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 2 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator 2 | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 4 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last | `load` an Bus 3 | P = 0,1 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

### Datenstruktur

In [103]:
net, idx = bn.build_110kV_net()

In [104]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,110.0,b,None,True,None
1,Bus 2,20.0,b,None,True,None
2,Bus 3,0.4,b,None,True,None
3,Bus 4,0.4,b,None,True,None


In [105]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo 1,25 MVA 110/20 kV,0,1,25.0,110.0,20.0,12.0,0.410,14.00,...,0.0,0.0,Ratio,<NA>,False,1,1.0,True,YNd5,False
1,Trafo 2,0.4 MVA 20/0.4 kV,1,2,0.4,20.0,0.4,6.0,1.425,1.35,...,0.0,0.0,Ratio,<NA>,False,1,1.0,True,Dyn5,False


In [106]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,2,3,0.1,0.642,0.083,210.0,0.0,0.142,1.0,1,cs,True,None


In [107]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load,3,0.05,0.005,0.0,0.0,0.0,0.0,NaN,1.0,True,wye


### Ergebnisse

In [108]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.067078,-0.016452
1,1.019939,-150.015699,0.000000,0.000000
2,1.017368,59.578280,0.000000,0.000000
3,0.996984,59.545082,0.050000,0.005000


In [109]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.067078,0.016452,-0.052513,-0.005511,0.014565,0.010941,0.000355,0.001494,1.020000,0.000000,1.019939,-150.015699,0.270847
1,0.052513,0.005511,-0.051019,-0.005131,0.001494,0.000380,0.001494,0.072748,1.019939,-150.015699,1.017368,59.578280,12.942309


In [110]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.051019,0.005131,-0.05,-0.005,0.001019,0.000131,0.072748,0.072748,0.072748,1.017368,59.57828,0.996984,59.545082,51.231076


In [111]:
net.res_load

,p_mw,q_mvar
0,0.05,0.005


## Aufgabe 6.2
Zwei lasten

### Netzwerk

Externes Netz → Bus 1 (20 kV) → Transformator → Bus 2 (0,4 kV) → Kabel → Bus 3 (0,4 kV) → Last 1 | Last 2

An Bus 3 wurde eine weitere Last `load2` angeschlossen.

| Komponente | pandapower-Element | Wichtige Daten | Funktion |
|---|---|---|---|
| Externes Netz | `ext_grid` an Bus 1 | `vm_pu` = 1,02, Winkel 0° | Vorgelagertes Mittelspannungsnetz als starre Spannungsquelle. Es ist der Slack-Bus: Er gibt Spannung und Winkelreferenz vor und gleicht die Leistungsbilanz (Last + Verluste) aus. |
| Bus 1 | `bus`, 20 kV | Nennspannung 20 kV | Mittelspannungs-Sammelschiene, Oberspannungsseite des Trafos |
| Transformator 2 | `trafo`, Typ „0.4 MVA 20/0.4 kV“ | S_r = 0,4 MVA, v_k = 6 %, v_kr = 1,425 %, Schaltgruppe Dyn5, Stufensteller ±2 × 2,5 % auf der OS-Seite | Ortsnetztransformator, setzt 20 kV auf 0,4 kV um |
| Bus 2 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Niederspannungs-Sammelschiene der Ortsnetzstation |
| Kabel | `line`, Typ „NAYY 4x50 SE“ | 0,1 km, r = 0,642 Ω/km, x = 0,083 Ω/km, c = 210 nF/km, I_max = 142 A, 1 System | Niederspannungs-Erdkabel (Aluminium, 50 mm²) zum Verbraucher |
| Bus 3 | `bus`, 0,4 kV | Nennspannung 0,4 kV | Netzknoten am Ende des Kabels, Anschlusspunkt des Verbrauchers |
| Last 1 | `load1` an Bus 3 | P = 0.05 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |
| Last 2 | `load2` an Bus 3 | P = 0.02 MW, Q = 0,005 Mvar | Verbraucher als Konstantleistungslast (ZIP-Anteile = 0) |

### Datenstruktur

In [112]:
net, idx = bn.build_2_load_net()

In [113]:
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,20.0,b,None,True,None
1,Bus 2,0.4,b,None,True,None
2,Bus 3,0.4,b,None,True,None


In [114]:
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_step_degree,tap_pos,tap_changer_type,id_characteristic_table,tap_dependency_table,parallel,df,in_service,vector_group,oltc
0,Trafo,0.4 MVA 20/0.4 kV,0,1,0.4,20.0,0.4,6.0,1.425,1.35,...,0.0,0.0,Ratio,<NA>,False,1,1.0,True,Dyn5,False


In [115]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line,NAYY 4x50 SE,1,2,0.1,0.642,0.083,210.0,0.0,0.142,1.0,1,cs,True,None


In [116]:
net.load

,name,bus,p_mw,q_mvar,const_z_p_percent,const_i_p_percent,const_z_q_percent,const_i_q_percent,sn_mva,scaling,in_service,type
0,Load 1,2,0.05,0.005,0.0,0.0,0.0,0.0,NaN,1.0,True,wye
1,Load 2,2,0.02,0.005,0.0,0.0,0.0,0.0,NaN,1.0,True,wye


### Ergebnisse

In [117]:
pp.runpp(net)
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.020000,0.000000,-0.073645,-0.011027
1,1.015927,-150.565965,0.000000,0.000000
2,0.986942,-150.544191,0.070000,0.010000


In [118]:
net.res_trafo

,p_hv_mw,q_hv_mvar,p_lv_mw,q_lv_mvar,pl_mw,ql_mvar,i_hv_ka,i_lv_ka,vm_hv_pu,va_hv_degree,vm_lv_pu,va_lv_degree,loading_percent
0,0.073645,0.011027,-0.07206,-0.010265,0.001585,0.000762,0.002107,0.103412,1.02,0.0,1.015927,-150.565965,18.251463


In [119]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,0.07206,0.010265,-0.07,-0.01,0.00206,0.000265,0.103412,0.103412,0.103412,1.015927,-150.565965,0.986942,-150.544191,72.825633


In [120]:
net.res_load

,p_mw,q_mvar
0,0.05,0.005
1,0.02,0.005
